# TetraFT — Kaggle runs

**Attach datasets**
- `tetraft-code` — flat `.py` + this notebook (**refresh** after code changes)
- **`tetraft-fineweb-edu-400m`** — marathon train/val (or `…-50m` for scouts)
- **Sessions 2–16:** prior run pack Dataset `tetraft-heal-kl-trust-400m` (has `checkpoint-final` + ledger)

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first model download |

## Marathon (paper run) — set `SESSION = 1` … `16`

| SESSION | Steps stop | ≈ tok | Ckpt | Gate PPL |
|--------:|----------:|------:|------|----------|
| **1** | 6104 | 25M | **full** | &lt; 48.65 |
| **2** | 12208 | 50M | full | &lt; 34.38 |
| **4** | 24416 | 100M | full | &lt; 30 |
| **16** | 97664 | 400M | weights OK | stretch ≲ 23 |

**DNA:** `heal_kl_trust_400m` — trust STE s=1.0, α=0.3, T=2, no LoRA.

**After each hop:** Save Version → update Dataset `tetraft-heal-kl-trust-400m` from  
`/kaggle/working/heal_kl_trust_400m/` (ledger + `sessions/Sxx/` + checkpoint).

## Other sessions (scouts / legacy)

| SESSION | What |
|---------|------|
| **T** / **T0** | trust scouts @ 5M |
| **R5** | LoRA scout (done 48.38) |
| **L** | layer map |
| **A**/**B** | heal_kl_50m legacy |


In [ ]:
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

from config import SMOKE_PRESETS
assert "heal_kl_trust_400m" in SMOKE_PRESETS, "heal_kl_trust_400m missing — refresh tetraft-code"
assert "scout_kl_trust_a03_5m" in SMOKE_PRESETS
assert (code_root / "run_pack.py").is_file(), "run_pack.py missing — refresh tetraft-code"
print("presets ok:", sorted(SMOKE_PRESETS))

In [ ]:
from run_smoke import run_smoke
from run_layer_map import main as run_layer_map_main
from run_pack import (
    HORIZON_TRUST_400M,
    N_SESSIONS_TRUST_400M,
    RUN_ID_TRUST_400M,
    SESSION_MICRO_STEPS,
    ensure_run_root,
    find_prior_run_root,
    find_resume_checkpoint,
    session_gate_ppl,
    session_stop_step,
)
import argparse
import shutil
from pathlib import Path

# =============================================================================
# SESSION — marathon: integer 1..16  |  scouts: "T" "T0" "R5" "L" "A" "B" "P" "S"
# =============================================================================
SESSION = 1  # <-- 1..16 for heal_kl_trust_400m  |  or "T" / "R5" / ...

RUN_PACK_DIR = Path("/kaggle/working") / RUN_ID_TRUST_400M
CLEAR_WORKING_TRAIN = True  # clear per-hop train out; never touch attached input pack

# Scout-only knobs
SCOUT_ALPHA = 0.3
SCOUT_TEMPERATURE = 2.0
SCOUT_TAG = "a03_t2"
LAYER_MAP_CHECKPOINT = None
LAYER_MAP_FP_MASK_TOPK = 8
LAYER_MAP_SKIP_PPL = False

STE_MODE = None
TRUST_SOFTNESS = None
DISTILL_ALPHA = None
DISTILL_TEMPERATURE = None
PRE_RMS = None
NO_PRE_RMS = False
WEIGHT_CALIB = None
LORA_RANK = None
LORA_ALPHA = None
RUN_SESSION = None
RUN_PACK = None
NO_SAVE_OPTIMIZER = False

sess_raw = SESSION
if isinstance(sess_raw, int) or (isinstance(sess_raw, str) and str(sess_raw).isdigit()):
    SESSION_N = int(sess_raw)
    if SESSION_N < 1 or SESSION_N > N_SESSIONS_TRUST_400M:
        raise ValueError(f"SESSION must be 1..{N_SESSIONS_TRUST_400M}, got {SESSION_N}")

    PRESET = "heal_kl_trust_400m"
    MAX_STEPS = session_stop_step(SESSION_N)
    SAVE_OPTIMIZER = SESSION_N < N_SESSIONS_TRUST_400M
    NO_SAVE_OPTIMIZER = not SAVE_OPTIMIZER
    SKIP_SHOCK = SESSION_N > 1
    SKIP_ORIG = SESSION_N > 1
    RUN_SESSION = SESSION_N
    RUN_PACK = str(RUN_PACK_DIR)
    OUTPUT_DIR = f"/kaggle/working/train_{RUN_ID_TRUST_400M}_S{SESSION_N:02d}"

    # Seed pack from attached prior dataset (ledger history + ckpt)
    prior = find_prior_run_root()
    ensure_run_root(RUN_PACK_DIR)
    if prior is not None and prior.resolve() != RUN_PACK_DIR.resolve():
        print("prior run pack:", prior)
        # copy ledger / meta if working pack is empty
        for name in ("ledger.jsonl", "RUN_META.json", "baselines.json", "LATEST.json"):
            src, dst = prior / name, RUN_PACK_DIR / name
            if src.is_file() and not dst.is_file():
                shutil.copy2(src, dst)
                print("seeded", name)
        # copy prior sessions summaries (not full multi-GB ckpts except needed)
        src_sess = prior / "sessions"
        if src_sess.is_dir():
            for sdir in sorted(src_sess.glob("S*")):
                dst = RUN_PACK_DIR / "sessions" / sdir.name
                if dst.exists():
                    continue
                dst.mkdir(parents=True, exist_ok=True)
                for fn in ("session_summary.json", "smoke_results.json", "metrics.jsonl",
                           "linear_inventory.json", "checkpoint-final"):
                    fsrc = sdir / fn
                    if fsrc.is_file():
                        shutil.copy2(fsrc, dst / fn)
                print("seeded session", sdir.name)

    RESUME = None
    if SESSION_N > 1:
        ckpt = find_resume_checkpoint(RUN_PACK_DIR, SESSION_N)
        if ckpt is None and prior is not None:
            ckpt = find_resume_checkpoint(prior, SESSION_N)
        if ckpt is None:
            ckpt = Path(find_file("checkpoint-final"))
        RESUME = str(ckpt)
        print("resume from:", RESUME)

    gate = session_gate_ppl(SESSION_N)
    print(
        f"MARATHON S{SESSION_N:02d}/{N_SESSIONS_TRUST_400M} "
        f"max_steps={MAX_STEPS} horizon={HORIZON_TRUST_400M} "
        f"save_optimizer={SAVE_OPTIMIZER} gate_ppl<{gate}"
    )
    print("DNA: trust s=1.0 α=0.3 T=2 skip_GDN — publish pack after hop:", RUN_PACK_DIR)

elif str(sess_raw).upper() == "T":
    PRESET = "scout_kl_trust_a03_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    OUTPUT_DIR = "/kaggle/working/checkpoints_scout_kl_trust_a03_5m"
    print("T: trust+α0.3 @ 5M gate < 48.38")
elif str(sess_raw).upper() == "T0":
    PRESET = "scout_kl_trust_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    OUTPUT_DIR = "/kaggle/working/checkpoints_scout_kl_trust_5m"
    print("T0: trust-only α0.5 @ 5M")
elif str(sess_raw).upper() == "R5":
    PRESET = "scout_kl_r5_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    OUTPUT_DIR = "/kaggle/working/checkpoints_scout_kl_r5_5m"
elif str(sess_raw).upper() == "U":
    raise RuntimeError("SESSION=U bundle FAIL — use SESSION=1 marathon or T")
elif str(sess_raw).upper() == "L":
    ckpt = LAYER_MAP_CHECKPOINT or str(find_file("checkpoint-final"))
    out_lm = Path("/kaggle/working/layer_map_b")
    if out_lm.exists():
        shutil.rmtree(out_lm)
    out_lm.mkdir(parents=True, exist_ok=True)
    argv = [
        "--checkpoint", ckpt, "--preset", "heal_kl_50m",
        "--val-data", str(val_path), "--max-eval-batches", "20",
        "--fp-mask-topk", str(int(LAYER_MAP_FP_MASK_TOPK)),
        "--output-dir", str(out_lm),
    ]
    if LAYER_MAP_SKIP_PPL:
        argv.append("--skip-ppl")
    raise SystemExit(run_layer_map_main(argv))
elif str(sess_raw).upper() == "A":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 6104
    SAVE_OPTIMIZER = True
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_A"
elif str(sess_raw).upper() == "B":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 12207
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_B"
elif str(sess_raw).upper() == "P":
    PRESET = "polish_kl_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    OUTPUT_DIR = "/kaggle/working/checkpoints_polish_kl_5m"
elif str(sess_raw).upper() == "S":
    PRESET = "scout_kl_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = float(SCOUT_ALPHA)
    DISTILL_TEMPERATURE = float(SCOUT_TEMPERATURE)
    OUTPUT_DIR = f"/kaggle/working/checkpoints_scout_kl_{SCOUT_TAG}"
else:
    raise ValueError("SESSION must be 1..16 or T/T0/R5/L/A/B/P/S")

out = Path(OUTPUT_DIR)
if CLEAR_WORKING_TRAIN and out.exists():
    shutil.rmtree(out)
    print("cleared", out)
out.mkdir(parents=True, exist_ok=True)

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir=OUTPUT_DIR,
    seq_length=None,
    batch_size=None,
    max_steps=MAX_STEPS,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=SKIP_SHOCK,
    skip_orig=SKIP_ORIG,
    resume=RESUME,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    lr_scheduler=None,
    min_lr_ratio=None,
    schedule_max_steps=None,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    save_optimizer=bool(SAVE_OPTIMIZER) and not NO_SAVE_OPTIMIZER,
    no_save_optimizer=NO_SAVE_OPTIMIZER,
    skip_linear_attn=None,
    no_skip_linear_attn=False,
    distill_alpha=DISTILL_ALPHA,
    distill_temperature=DISTILL_TEMPERATURE,
    quant_reg_beta=None,
    pre_rms=PRE_RMS,
    no_pre_rms=NO_PRE_RMS,
    weight_calib=WEIGHT_CALIB,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    ste_mode=STE_MODE,
    trust_softness=TRUST_SOFTNESS,
    run_session=RUN_SESSION,
    run_pack_dir=RUN_PACK,
    run_id=RUN_ID_TRUST_400M if RUN_SESSION else None,
    seed=42,
    device_map="auto",
)
print(
    f"SESSION={SESSION} preset={PRESET} max_steps={MAX_STEPS} "
    f"resume={RESUME} out={OUTPUT_DIR} pack={RUN_PACK}"
)
results = run_smoke(ns)
keys = [
    "preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
    "loss_finite", "tokens_seen", "tokens_budget", "steps_ran",
    "resumed_step", "schedule_horizon_steps", "distill",
    "session_summary", "run_pack_dir",
]
print({k: results[k] for k in keys if k in results})
if "session_summary" in results:
    s = results["session_summary"]
    print(
        f"PAPER {s.get('session_tag')}: ppl_end={s.get('ppl_end')} "
        f"after/orig={s.get('after_over_orig')} gate<{s.get('gate_ppl')} → {s.get('gate_status')}"
    )
    print("Download/publish run pack:", results.get("run_pack_dir"))
    print("  → Kaggle Dataset tetraft-heal-kl-trust-400m (include sessions/ + ledger + LATEST)")
ppl = results.get("ppl_after_smoke")
if ppl is not None and RUN_SESSION is None:
    ref = results.get("ppl_original") or 17.67
    print(f"after/orig ≈ {ppl / ref:.3f}")


### After each marathon session

1. Confirm `gate_status` in logs / `session_summary.json`.
2. **Save Version** with output → update Dataset **`tetraft-heal-kl-trust-400m`** from  
   `/kaggle/working/heal_kl_trust_400m/`.
3. Next hop: attach that Dataset + code + FineWeb-400m; set `SESSION = k+1`.

Keep in the Dataset: `ledger.jsonl`, `LATEST.json`, all `sessions/S*/session_summary.json` + `metrics.jsonl`,  
and **at least the latest full `checkpoint-final`** (S1–S15).

### Paper figures (local, after S16)

```bash
python scripts/merge_run_pack.py path/to/heal_kl_trust_400m
python scripts/plot_heal_kl_trust_400m.py path/to/heal_kl_trust_400m
```
